In [ ]:
import duckdb
import pandas as pd

In [ ]:
con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(f"SET s3_region='ap-northeast-2';")

### Pandas ↔ DuckDB

In [21]:
df = pd.DataFrame({"id": [1, 2, 3],"name": ["A", "B", "C"],"value": [10, 20, 30]})
con.register("df_tbl", df)
result = con.execute("SELECT name, value * 2 AS value2 FROM df_tbl WHERE value >= 20").df()
print(result)

  name  value2
0    B      40
1    C      60


### SQL 결과 → Pandas

In [22]:
out_df = con.execute("SELECT sum(value) AS total FROM df_tbl").df()
print(out_df)

   total
0   60.0


### S3 Parquet 파일 읽기

In [ ]:
s3_url = "s3://mmix-prod-dataengineer-datalakehouse/streaming/mmix_mysql_cdc/mmix_departments_iceberg_sink/data/*.parquet"
dataframe = con.execute(f"""SELECT * FROM read_parquet('{s3_url}') LIMIT 100""").df()
len(dataframe)